In [6]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [8]:
def load_file_data(path):
    data_list = []
    if not path:
        raise ValueError("path가 비어 있습니다. 파일 경로 리스트를 전달해야 합니다.")
    if not isinstance(path, list):
        raise TypeError("path는 list 타입이어야 합니다.")

    for file_ in path:
        filename = os.path.basename(file_).split('.')[0]
        try:
            with open(file_, 'r', encoding='latin1') as f:
                text_content = f.read()
            data_list.append({'topic': filename, 'text': text_content})
        except FileNotFoundError:
            print(f"Error: {file_} not found.")
        except Exception as e:
            print(f"An error occurred reading {file_}: {e}")

    return pd.DataFrame(data_list)


# 실제 데이터 로드
path = r'../data'
all_files = glob.glob(os.path.join(path, "*.data"))   # 확장자 확인 필요
df = load_file_data(all_files)


print(f"전체 데이터 크기: {df.shape}")
print(f"토픽 개수: {df['topic'].nunique()}")
print(f"\n샘플 데이터:")
print(df.head(10))

전체 데이터 크기: (51, 2)
토픽 개수: 51

샘플 데이터:
                             topic  \
0    accuracy_garmin_nuvi_255W_gps   
1   bathroom_bestwestern_hotel_sfo   
2       battery-life_amazon_kindle   
3       battery-life_ipod_nano_8gb   
4      battery-life_netbook_1005ha   
5            buttons_amazon_kindle   
6        comfort_honda_accord_2008   
7        comfort_toyota_camry_2007   
8  directions_garmin_nuvi_255W_gps   
9     display_garmin_nuvi_255W_gps   

                                                text  
0  , and is very, very accurate .\n but for the m...  
1   The room was not overly big, but clean and ve...  
2   After I plugged it in to my USB hub on my com...  
3   short battery life  I moved up from an 8gb .\...  
4  6GHz 533FSB cpu, glossy display, 3, Cell 23Wh ...  
5  I thought it would be fitting to christen my K...  
6   Drivers seat not comfortable, the car itself ...  
7   Ride seems comfortable and gas mileage fairly...  
8   You also get upscale features like spoken di


토픽별 데이터 수:
topic
accuracy_garmin_nuvi_255W_gps      1
service_bestwestern_hotel_sfo      1
price_holiday_inn_london           1
quality_toyota_camry_2007          1
rooms_bestwestern_hotel_sfo        1
rooms_swissotel_chicago            1
room_holiday_inn_london            1
satellite_garmin_nuvi_255W_gps     1
screen_garmin_nuvi_255W_gps        1
screen_ipod_nano_8gb               1
screen_netbook_1005ha              1
seats_honda_accord_2008            1
service_holiday_inn_london         1
performance_netbook_1005ha         1
service_swissotel_hotel_chicago    1
Name: count, dtype: int64

필터링 후 데이터: (10, 2)
선택된 토픽: 10개

선택된 토픽 목록:
  1. accuracy_garmin_nuvi_255W_gps: 1개
  2. service_bestwestern_hotel_sfo: 1개
  3. price_holiday_inn_london: 1개
  4. quality_toyota_camry_2007: 1개
  5. rooms_bestwestern_hotel_sfo: 1개
  6. rooms_swissotel_chicago: 1개
  7. room_holiday_inn_london: 1개
  8. satellite_garmin_nuvi_255W_gps: 1개
  9. screen_garmin_nuvi_255W_gps: 1개
  10. screen_ipod_nano_8gb: 1개

In [ ]:
# 텍스트 길이 분포 확인
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("\n텍스트 통계:")
print(df[['text_length', 'word_count']].describe())

# 시각화
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df['text_length'].hist(bins=20, ax=ax[0])
ax[0].set_title('텍스트 길이 분포')
df['word_count'].hist(bins=20, ax=ax[1])
ax[1].set_title('단어 수 분포')
plt.show()

In [ ]:
# 토픽에서 제품 카테고리 추출
def extract_category(topic):
    """
    토픽명에서 제품 카테고리 추출
    예: accuracy_garmin_nuvi_255W_gps -> garmin (GPS)
    """
    parts = topic.split('_')
    if 'garmin' in topic:
        return 'GPS'
    elif 'hotel' in topic or 'inn' in topic or 'swissotel' in topic:
        return 'Hotel'
    elif 'honda' in topic or 'toyota' in topic or 'camry' in topic:
        return 'Car'
    elif 'kindle' in topic or 'ipod' in topic or 'netbook' in topic:
        return 'Electronics'
    else:
        return 'Other'

df['category'] = df['topic'].apply(extract_category)

print("\n카테고리별 분포:")
print(df['category'].value_counts())

In [ ]:
# 토픽명에서 특성(feature) 추출
df['feature'] = df['topic'].str.split('_').str[0]

print("\n주요 특성별 분포:")
print(df['feature'].value_counts().head(10))

# 예상 출력:
# battery-life    3
# screen          3
# rooms           2
# comfort         2
# service         2

In [ ]:
# 결측치 확인
print("\n결측치 확인:")
print(df.isnull().sum())

# 중복 데이터 확인
print(f"\n중복 토픽: {df['topic'].duplicated().sum()}개")

# 빈 텍스트 확인
empty_texts = df[df['text'].str.strip() == '']
print(f"\n빈 텍스트: {len(empty_texts)}개")

if len(empty_texts) > 0:
    print("빈 텍스트 토픽:")
    print(empty_texts['topic'].tolist())
